# 03 - MLflow Tracking and Joblib Export

This notebook trains the same reproducible pipeline, records the experiment with MLflow, and saves the complete sklearn pipeline with Joblib for the Flask application.

Everything is local: no cloud MLflow account is required.

In [1]:
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
dataset = fetch_california_housing(as_frame=True)
df = dataset.frame.copy()

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

In [3]:
n_estimators = 120
random_state = 42

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )),
])

## 1. Configure Local MLflow Tracking

In [5]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_tracking_uri("sqlite:///./mlflow.db")
mlflow.set_experiment("end-to-end-california-housing")

2026/09/21 21:42:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/21 21:42:13 INFO mlflow.store.db.utils: Updating database tables
2026/09/21 21:42:17 INFO mlflow.tracking.fluent: Experiment with name 'end-to-end-california-housing' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/user/Desktop/End-to-End-ML-Pipeline/notebooks/mlruns/1', creation_time=1790007137272, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790007137272, lifecycle_stage='active', name='end-to-end-california-housing', tags={}, trace_location=None, workspace='default'>

## 2. Train and Track the Experiment

In [7]:
from mlflow.sklearn import log_model

with mlflow.start_run():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("random_state", random_state)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    log_model(
        pipeline,
        artifact_path="california_housing_pipeline",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree",
        ],
    )

    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

2026/09/21 21:43:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/21 21:44:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


MAE : 0.3276
RMSE: 0.5052
R²  : 0.8053


## 3. Export with Joblib

The complete pipeline is saved as one artifact. This includes imputation, scaling, and the Random Forest estimator.

In [8]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "california_housing_pipeline.joblib"
joblib.dump(pipeline, model_path)

print(f"Saved model to: {model_path.resolve()}")

Saved model to: C:\Users\user\Desktop\End-to-End-ML-Pipeline\models\california_housing_pipeline.joblib


## 4. Test the Saved Artifact

In [9]:
loaded_pipeline = joblib.load(model_path)

sample_prediction = loaded_pipeline.predict(X_test.iloc[[0]])[0]
print("Prediction from saved pipeline:", round(float(sample_prediction), 4))

Prediction from saved pipeline: 0.5012


## 5. MLflow UI

From the repository root, run:

```bash
mlflow ui
```

MLflow will provide a local browser address where the run parameters and metrics can be inspected.

The saved Joblib pipeline is the artifact used by `app.py`.